<a href="https://colab.research.google.com/github/w3aarush/DR_Classification_NIT_MCA_Project/blob/main/EfficientNet_KNN_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
# from tensorflow.keras.layers.experimental import preprocessing
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Attention
# from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
import matplotlib.pyplot as plt

In [2]:
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S, preprocess_input
from google.colab.patches import cv2_imshow
import pandas as pd
import numpy as np
import seaborn as sns
# import imutils
import time
import cv2
from cuml import SVC # for python 3.11
# from sklearn.svm import SVC

In [3]:
# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()

# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download DATASET (not competition)
!kaggle datasets download -d mariaherrerot/aptos2019

# Unzip
!unzip -q aptos2019.zip -d aptos2019

# Check files
!ls aptos2019

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mariaherrerot/aptos2019
License(s): unknown
100% 8.01G/8.01G [01:18<00:00, 109MB/s]

test.csv  test_images  train_1.csv  train_images  valid.csv  val_images


In [4]:
base_dir = '/content/aptos2019'
train_dir = '/content/aptos2019/train_images/train_images/'
validation_dir = '/content/aptos2019/val_images/val_images/'
test_dir = '/content/aptos2019/test_images/test_images/'

In [5]:
import os

In [6]:
print(os.listdir(train_dir))
print(os.listdir(validation_dir))
print(os.listdir(test_dir))

['ae975c43bd8b.png', 'ba0107fb1bfd.png', 'cb0cc98d7e35.png', '910bfd38e2f5.png', 'da9262d9f5d9.png', '86baef833ae0.png', '2682e6da9050.png', '42af7282349b.png', 'd4f32b9c07df.png', 'b3c0c3330278.png', 'c67117c6ab3b.png', 'b460ca9fa26f.png', '39fd8ef3a45c.png', '96a9706b8534.png', '65e6f1bd9875.png', 'd1ca85af57c9.png', '6d6fcf49e515.png', '8114d6a160df.png', 'e322acd46152.png', '3ca8be3b40d6.png', '3b232b394e4f.png', '1e9224ccca95.png', 'bf811911acf9.png', 'df6d13d04da1.png', '874f8c1929f6.png', '1ca62b3e4fd3.png', 'cae51154e1ce.png', '76516f828d88.png', '2b07790a2422.png', '7828dd083cdc.png', 'e0b5a982a018.png', '2ef955d6d9ff.png', '929cd3867815.png', '7d37a2939f12.png', '6af071b0ac6e.png', '7a0cff4c24b2.png', 'b5bf7b84fc66.png', '3f5b4c2948e8.png', '4beeca5cc859.png', '4246ed634f25.png', '4ef16a53d899.png', '4c3c1ed09771.png', '94ef1d14597f.png', '8714d17bb6da.png', '72595230840c.png', '7663aba8d762.png', 'cb02bb47fdc5.png', 'b10fca20c885.png', '69df7ade0575.png', '69b3a00927fc.png',

In [7]:
NUM_CLASSES = 5
epochs = 20

In [8]:
BATCH_SIZE = 32

In [9]:
# img_augmentation = Sequential(
#     [
#         tf.keras.layers.RandomRotation(factor=(-0.15, 0.15)),
#         tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
#         tf.keras.layers.RandomFlip(),
#         tf.keras.layers.RandomContrast(factor=0.1),
#     ],
#     name="img_augmentation",
# )

This code defines an image augmentation pipeline using Keras's Sequential model. It applies a series of random transformations to input images:

RandomRotation(factor=(-0.15, 0.15)): Randomly rotates images by an angle within the range of -15% to +15% of 2π radians.
RandomTranslation(height_factor=0.1, width_factor=0.1): Randomly shifts images horizontally and vertically by up to 10% of their width and height, respectively.
RandomFlip(): Randomly flips images horizontally (left-right).
RandomContrast(factor=0.1): Randomly adjusts the contrast of images by a factor within the range of [1 - 0.1, 1 + 0.1] (i.e., [0.9, 1.1]).
This img_augmentation pipeline is typically used during model training to artificially increase the size and diversity of the training dataset, helping to improve the model's generalization capabilities and reduce overfitting.

In [42]:
def extract_EfficientNetV2S_feature_map():
    IMG_SIZE = (224, 224)
    # Load the EfficientNetV2S base model without the top (classification) layer
    base_model = EfficientNetV2S(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')

    # Freeze the base model layers
    base_model.trainable = False

    # Create a new model on top of the base model
    inputs = keras.Input(shape=IMG_SIZE + (3,))
   # x = img_augmentation(inputs)
    x = preprocess_input(inputs) # EfficientNetV2S expects inputs in the [-1, 1] range after preprocessing
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    feature_map = layers.Dense(64, activation='relu')(x)
    # outputs = layers.Dense(5, activation='softmax')(x)
    model = keras.Model(inputs, feature_map)

    # Compile the model
    # model.compile(
    #     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    #     loss='categorical_crossentropy',
    #     metrics=['accuracy']
    # )
    return model

In [92]:
def build_model_for_training():
    # Use the existing feature map logic
    IMG_SIZE = (224, 224)
    base_model = EfficientNetV2S(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base_model.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    # The 'feature_map' layer we want to optimize
    feature_layer = layers.Dense(64, activation='relu', name='feature_extraction_layer')(x)

    # The 'classification_head' needed to calculate the cost function (Loss)
    # This is what allows model.fit() to actually update the weights
    outputs = layers.Dense(5, activation='softmax')(feature_layer)

    model = keras.Model(inputs, outputs)
    return model

In [43]:
def unfreeze_model(model):
    # unfreeze the top 10 layers while leaving BatchNorm layers frozen for fine-tuning
    for layer in model.layers[-10:]:
        if not isinstance(layer, layers.BatchNormalization):
            print("executed")
            layer.trainable = True
    # optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    # model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [44]:
def test_model(model, test_batches):
    test_labels = test_batches.classes
    print("Test Lables", test_labels)
    print(test_batches.class_indices)

    # predictions = model.predict(test_batches, step=len(test_batches), verbose=0)
    predictions = model.predict(test_batches, verbose=0)


    acc = 0
    for i in range(len(test_labels)):
        actual_class = test_labels[i]
        if predictions[i][actual_class] > 0.5:
            acc += 1
    print('Accuracy:', (acc/len(test_labels))*100, "% ")
    # Convert predictions to discrete class labels for classification_report
    predicted_labels = np.argmax(predictions, axis=1)
    print('Classification Report:', classification_report(test_batches.labels, predicted_labels))

In [45]:
def load_data():
    train = pd.read_csv('/content/aptos2019/train_1.csv', encoding='utf-8')
    test = pd.read_csv('/content/aptos2019/test.csv', encoding='utf-8')
    valid = pd.read_csv('/content/aptos2019/valid.csv')

    train_dir = '/content/aptos2019/train_images/train_images/'
    test_dir = '/content/aptos2019/test_images/test_images/'
    valid_dir = '/content/aptos2019/val_images/val_images/'

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    train['train_images'] = train['id_code'] + '.png'
    test['test_images'] = test['id_code'] + '.png'
    valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(str)
    # train['target'] = [1 if x >= 1 else 0 for x in train['diagnosis']]
    # train['target'] = train['target'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)
    # test['target'] = [1 if x >= 1 else 0 for x in test['diagnosis']]
    # test['target'] = test['target'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)
    # valid['target'] = [1 if x >= 1 else 0 for x in valid['diagnosis']]
    # valid['target'] = valid['target'].astype(str)

    return train, test, valid

In [46]:
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis = 0)
    return preprocess_input(img_array)

In [47]:
model = extract_EfficientNetV2S_feature_map()

In [48]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        81,984 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,413,344 (77.87 MB)

 Trainable params: 81,984 (320.25 KB)

 Non-trainable params: 20,331,360 (77.56 MB)

In [49]:
train_df, test_df, valid_df = load_data()

In [94]:
# Initialize all data generators in the global scope
train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
    dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)

valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
    dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)

test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
    dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

Found 2930 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.


In [51]:
feature_maps = model.predict(train_batches)

92/92 ━━━━━━━━━━━━━━━━━━━━ 345s 4s/step


In [57]:
type(feature_maps)

numpy.ndarray

In [1]:
# Instantiate the model with the classification head
train_model = build_model_for_training()

# Compile the model
train_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5)

# Start training
history = train_model.fit(
    train_batches,
    epochs=epochs,
    validation_data=valid_batches,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

NameError: name 'build_model_for_training' is not defined

# Preparing DataFrame

In [62]:
feature_df = pd.DataFrame(feature_maps)
feature_df.columns = [f'X{i}' for i in range(feature_df.shape[1])]
# feature_df['diagnosis'] = train_df['diagnosis']
y = train_df['diagnosis']

In [63]:
feature_df.head()

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X54,X55,X56,X57,X58,X59,X60,X61,X62,X63
0,0.414258,0.054488,0.000000,0.504273,0.957093,0.000000,0.0,0.508730,0.0,0.271118,...,0.0,0.000000,0.0,0.426434,0.0,0.0,0.0,0.876927,0.045296,0.899285
1,0.287033,0.013695,0.128387,0.026678,1.324446,0.079313,0.0,0.596296,0.0,0.000000,...,0.0,0.051661,0.0,0.778544,0.0,0.0,0.0,0.701478,0.402761,0.987179
2,0.146794,0.059688,0.153535,0.499980,0.742206,0.058301,0.0,0.353486,0.0,0.000000,...,0.0,0.325450,0.0,0.701647,0.0,0.0,0.0,0.615544,0.000000,0.930663
3,0.000000,0.535488,0.000000,0.591791,0.751496,0.000000,0.0,0.216733,0.0,0.105048,...,0.0,0.429889,0.0,0.651235,0.0,0.0,0.0,0.252735,0.434725,0.839333
4,0.000000,0.670318,0.010228,0.327081,0.717496,0.000000,0.0,0.578692,0.0,0.153530,...,0.0,0.000000,0.0,0.735262,0.0,0.0,0.0,0.505649,0.367456,1.042519


In [65]:
y.value_counts()

,count
diagnosis,
0,1434
2,808
1,300
4,234
3,154


# KNN

In [66]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=20)

In [67]:
knn.fit(feature_df, y)

KNeighborsClassifier(n_neighbors=20)

In [68]:
result = knn.predict(feature_df)
accu_score = accuracy_score(y, result)
print(accu_score)

0.5020477815699659


In [79]:
inst_1 = knn.predict(feature_df.iloc[[1]])
# feature_df.iloc[[0]]

In [80]:
inst_1

array(['2'], dtype=object)

In [81]:
y[1]

'1'

# Previous Code

In [ ]:
if __name__ == "__main__":
    train_df, test_df, valid_df = load_data()
    model = extract_EfficientNetV2S_feature_map()
    unfreeze_model(model)

    train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)
    history = model.fit(train_batches, epochs=epochs, validation_data=valid_batches, verbose=1,callbacks=[early_stopping,reduce_lr])